In [2]:
!pip install pandas pyarrow deltalake fastavro

   ---------------------------------------- 0.0/44.1 MB ? eta -:--:--
   - -------------------------------------- 1.6/44.1 MB 10.0 MB/s eta 0:00:05
   -- ------------------------------------- 2.9/44.1 MB 7.6 MB/s eta 0:00:06
   --- ------------------------------------ 4.2/44.1 MB 7.8 MB/s eta 0:00:06
   ---- ----------------------------------- 5.5/44.1 MB 7.1 MB/s eta 0:00:06
   ----- ---------------------------------- 6.0/44.1 MB 6.1 MB/s eta 0:00:07
   ----- ---------------------------------- 6.3/44.1 MB 5.4 MB/s eta 0:00:07
   ------ --------------------------------- 6.8/44.1 MB 4.8 MB/s eta 0:00:08
   ------ --------------------------------- 7.3/44.1 MB 4.5 MB/s eta 0:00:09
   ------- -------------------------------- 8.1/44.1 MB 4.4 MB/s eta 0:00:09
   ------- -------------------------------- 8.7/44.1 MB 4.3 MB/s eta 0:00:09
   -------- ------------------------------- 9.2/44.1 MB 4.0 MB/s eta 0:00:09
   -------- ------------------------------- 9.4/44.1 MB 3.9 MB/s eta 0:00:09
   --


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# CONVERSOR LOCAL SEM SPARK
# CSV -> 1 PARQUET POR TABELA + 1 AVRO POR TABELA + DELTA
#
# Estrutura:
# datasets/
# ├── csv/
# ├── parquet/
# ├── avro/
# └── delta/
# ============================================================

import re
import shutil
import time
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from fastavro import writer as avro_writer
from fastavro import parse_schema

try:
    from deltalake import write_deltalake
except ImportError:
    from deltalake.writer import write_deltalake


# ============================================================
# CONFIGURAÇÕES
# ============================================================

BASE_PATH = Path("datasets").resolve()

CSV_DIR = BASE_PATH / "csv"
PARQUET_DIR = BASE_PATH / "parquet"
AVRO_DIR = BASE_PATH / "avro"
DELTA_DIR = BASE_PATH / "delta"

GERAR_PARQUET = True
GERAR_AVRO = True
GERAR_DELTA = True

ADICIONAR_COLUNAS_CONTROLE = True

# Para 500 mil linhas por tabela, 100 mil é um bom equilíbrio.
# Se faltar memória, use 50_000.
CHUNK_SIZE = 100_000

# Para teste rápido:
# LIMITE_ARQUIVOS = 1
LIMITE_ARQUIVOS = None

CSV_READ_OPTIONS = {
    "dtype": str,
    "keep_default_na": False,
    "na_filter": False,
    "encoding": "utf-8",
}


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def limpar_nome_tabela(nome_arquivo: str) -> str:
    """
    Exemplo:
    01_unidade_restaurante.csv -> unidade_restaurante
    """
    stem = Path(nome_arquivo).stem
    return re.sub(r"^\d+_", "", stem)


def listar_csvs() -> list[Path]:
    if not CSV_DIR.exists():
        raise FileNotFoundError(
            f"Pasta não encontrada: {CSV_DIR}\n"
            f"Crie a pasta datasets/csv e coloque os CSVs dentro."
        )

    arquivos = sorted(
        [
            arquivo
            for arquivo in CSV_DIR.iterdir()
            if arquivo.is_file() and arquivo.suffix.lower() == ".csv"
        ],
        key=lambda p: p.name
    )

    if LIMITE_ARQUIVOS is not None:
        arquivos = arquivos[:LIMITE_ARQUIVOS]

    return arquivos


def preparar_pasta(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def remover_arquivo_se_existir(path: Path) -> None:
    if path.exists():
        path.unlink()


def criar_diretorios_base() -> None:
    if GERAR_PARQUET:
        PARQUET_DIR.mkdir(parents=True, exist_ok=True)

    if GERAR_AVRO:
        AVRO_DIR.mkdir(parents=True, exist_ok=True)

    if GERAR_DELTA:
        DELTA_DIR.mkdir(parents=True, exist_ok=True)


def normalizar_dataframe_bronze(df: pd.DataFrame, csv_path: Path) -> pd.DataFrame:
    """
    Mantém tudo como string.
    Isso é adequado para Bronze.
    A Silver depois converte datas, números, CPFs, telefones etc.
    """

    df = df.fillna("")

    for coluna in df.columns:
        df[coluna] = df[coluna].astype(str)

    if ADICIONAR_COLUNAS_CONTROLE:
        ingestion_ts = datetime.now(timezone.utc).isoformat()

        df["_bronze_source_file"] = csv_path.name
        df["_bronze_ingestion_ts"] = ingestion_ts
        df["_bronze_layer"] = "bronze"

    return df


def validar_nome_avro(nome: str) -> str:
    """
    Avro exige nomes de campos compatíveis.
    """
    nome_limpo = re.sub(r"[^A-Za-z0-9_]", "_", nome)

    if not re.match(r"^[A-Za-z_]", nome_limpo):
        nome_limpo = f"col_{nome_limpo}"

    return nome_limpo


def ajustar_colunas_para_avro(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ajusta nomes das colunas apenas para o Avro.
    Parquet e Delta mantêm os nomes originais.
    """

    novos_nomes = []
    nomes_usados = set()

    for coluna in df.columns:
        nome_avro = validar_nome_avro(coluna)
        nome_base = nome_avro
        contador = 1

        while nome_avro in nomes_usados:
            nome_avro = f"{nome_base}_{contador}"
            contador += 1

        nomes_usados.add(nome_avro)
        novos_nomes.append(nome_avro)

    df_avro = df.copy()
    df_avro.columns = novos_nomes

    return df_avro


def criar_schema_avro(df: pd.DataFrame, nome_tabela: str) -> dict:
    campos = []

    for coluna in df.columns:
        campos.append({
            "name": coluna,
            "type": "string",
            "default": ""
        })

    return {
        "type": "record",
        "name": validar_nome_avro(nome_tabela),
        "namespace": "rh_bronze",
        "fields": campos
    }


def mostrar_configuracao() -> None:
    print("=" * 80)
    print("CONVERSOR LOCAL SEM SPARK")
    print("=" * 80)
    print(f"CSV_DIR...............: {CSV_DIR}")
    print(f"PARQUET_DIR...........: {PARQUET_DIR}")
    print(f"AVRO_DIR..............: {AVRO_DIR}")
    print(f"DELTA_DIR.............: {DELTA_DIR}")
    print("-" * 80)
    print(f"GERAR_PARQUET.........: {GERAR_PARQUET}")
    print(f"GERAR_AVRO............: {GERAR_AVRO}")
    print(f"GERAR_DELTA...........: {GERAR_DELTA}")
    print(f"CHUNK_SIZE............: {CHUNK_SIZE}")
    print(f"LIMITE_ARQUIVOS.......: {LIMITE_ARQUIVOS}")
    print("=" * 80)


# ============================================================
# CONVERSÃO DE UMA TABELA
# ============================================================

def converter_arquivo_csv(csv_path: Path, indice: int, total_arquivos: int) -> None:
    inicio_tabela = time.perf_counter()

    nome_arquivo = csv_path.name
    nome_tabela = limpar_nome_tabela(nome_arquivo)

    parquet_file = PARQUET_DIR / f"{nome_tabela}.parquet"
    avro_file = AVRO_DIR / f"{nome_tabela}.avro"
    delta_tabela_dir = DELTA_DIR / nome_tabela

    print(f"[{indice}/{total_arquivos}] Processando: {nome_arquivo}")
    print(f"Tabela lógica: {nome_tabela}")

    if GERAR_PARQUET:
        remover_arquivo_se_existir(parquet_file)

    if GERAR_AVRO:
        remover_arquivo_se_existir(avro_file)

    if GERAR_DELTA:
        preparar_pasta(delta_tabela_dir)

    total_linhas = 0

    parquet_writer = None
    avro_schema_parseado = None
    primeiro_chunk_delta = True
    primeiro_chunk_avro = True

    leitor_csv = pd.read_csv(
        csv_path,
        chunksize=CHUNK_SIZE,
        **CSV_READ_OPTIONS
    )

    try:
        for chunk_idx, df_chunk in enumerate(leitor_csv, start=1):
            inicio_chunk = time.perf_counter()

            df_chunk = normalizar_dataframe_bronze(df_chunk, csv_path)
            total_linhas += len(df_chunk)

            # ----------------------------
            # Parquet único por tabela
            # ----------------------------
            if GERAR_PARQUET:
                arrow_table = pa.Table.from_pandas(df_chunk, preserve_index=False)

                if parquet_writer is None:
                    parquet_writer = pq.ParquetWriter(
                        parquet_file,
                        arrow_table.schema,
                        compression="snappy"
                    )

                parquet_writer.write_table(arrow_table)

            # ----------------------------
            # Delta Lake
            # Delta SEMPRE será pasta/tabela
            # ----------------------------
            if GERAR_DELTA:
                arrow_table_delta = pa.Table.from_pandas(df_chunk, preserve_index=False)

                modo_delta = "overwrite" if primeiro_chunk_delta else "append"

                write_deltalake(
                    str(delta_tabela_dir),
                    arrow_table_delta,
                    mode=modo_delta
                )

                primeiro_chunk_delta = False

            # ----------------------------
            # Avro único por tabela
            # ----------------------------
            if GERAR_AVRO:
                df_avro = ajustar_colunas_para_avro(df_chunk)

                if avro_schema_parseado is None:
                    schema_avro = criar_schema_avro(df_avro, nome_tabela)
                    avro_schema_parseado = parse_schema(schema_avro)

                registros = df_avro.to_dict(orient="records")

                if primeiro_chunk_avro:
                    with open(avro_file, "wb") as out:
                        avro_writer(out, avro_schema_parseado, registros)

                    primeiro_chunk_avro = False

                else:
                    # Append no mesmo arquivo Avro
                    with open(avro_file, "a+b") as out:
                        avro_writer(out, None, registros)

            fim_chunk = time.perf_counter()
            tempo_chunk = fim_chunk - inicio_chunk

            print(
                f"  chunk {chunk_idx:03d} | "
                f"linhas acumuladas: {total_linhas:,} | "
                f"tempo chunk: {tempo_chunk:.2f}s"
            )

    finally:
        if parquet_writer is not None:
            parquet_writer.close()

    fim_tabela = time.perf_counter()
    tempo_tabela_min = (fim_tabela - inicio_tabela) / 60

    print(f"OK: {nome_tabela}")
    print(f"Linhas processadas: {total_linhas:,}")
    print(f"Tempo da tabela: {tempo_tabela_min:.2f} min")

    if GERAR_PARQUET:
        print(f"Parquet único: {parquet_file}")

    if GERAR_AVRO:
        print(f"Avro único:    {avro_file}")

    if GERAR_DELTA:
        print(f"Delta tabela:  {delta_tabela_dir}")

    print("-" * 80)


# ============================================================
# CONVERSÃO PRINCIPAL
# ============================================================

def converter_todos_csvs() -> None:
    inicio_geral = time.perf_counter()

    criar_diretorios_base()

    arquivos_csv = listar_csvs()

    if not arquivos_csv:
        raise FileNotFoundError(f"Nenhum CSV encontrado em: {CSV_DIR}")

    mostrar_configuracao()

    print(f"CSV encontrados: {len(arquivos_csv)}")
    print("-" * 80)

    for indice, csv_path in enumerate(arquivos_csv, start=1):
        converter_arquivo_csv(csv_path, indice, len(arquivos_csv))

    fim_geral = time.perf_counter()
    tempo_total_min = (fim_geral - inicio_geral) / 60

    print("CONVERSÃO FINALIZADA COM SUCESSO")
    print(f"Tempo total: {tempo_total_min:.2f} min")


# ============================================================
# EXECUTAR
# ============================================================

converter_todos_csvs()

CONVERSOR LOCAL SEM SPARK
CSV_DIR...............: C:\Users\William\Desktop\Data-Mesh-Organization\Recursos Humanos\Datasets\csv
PARQUET_DIR...........: C:\Users\William\Desktop\Data-Mesh-Organization\Recursos Humanos\Datasets\parquet
AVRO_DIR..............: C:\Users\William\Desktop\Data-Mesh-Organization\Recursos Humanos\Datasets\avro
DELTA_DIR.............: C:\Users\William\Desktop\Data-Mesh-Organization\Recursos Humanos\Datasets\delta
--------------------------------------------------------------------------------
GERAR_PARQUET.........: True
GERAR_AVRO............: True
GERAR_DELTA...........: True
CHUNK_SIZE............: 100000
LIMITE_ARQUIVOS.......: None
CSV encontrados: 12
--------------------------------------------------------------------------------
[1/12] Processando: 01_unidade_restaurante.csv
Tabela lógica: unidade_restaurante
  chunk 001 | linhas acumuladas: 100,000 | tempo chunk: 1.94s
  chunk 002 | linhas acumuladas: 200,000 | tempo chunk: 1.82s
  chunk 003 | linhas acu